In [ ]:
!pip install datasets transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from transformers import Trainer,TrainingArguments
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader
import torch

In [ ]:
from datasets import load_dataset

ds = load_dataset("dair-ai/emotion", "split")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
def process(ex):

  return tokenizer(ex['text'],truncation=True,padding=True)

In [ ]:
tokenized_dataset = ds.map(process,batched=True)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
datacollator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased",num_labels=6)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
training_dataset = DataLoader(
    dataset=tokenized_dataset['train'],
    batch_size=16,
    collate_fn=datacollator,
    shuffle=True,
)

testing_dataset = DataLoader(
    dataset=tokenized_dataset['test'],
    batch_size=16,
    collate_fn=datacollator,
    shuffle=True,
)


In [ ]:
train_arguments = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    no_cuda=False,
    fp16=True,
    fp16_opt_level='O1'
)

In [ ]:
trainer = Trainer(
    model=model,
    args=train_arguments,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=datacollator,
)

<ipython-input-18-878ae7f271bd>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

Step,Training Loss
500,0.639800
1000,0.247500
1500,0.156400
2000,0.139300
2500,0.110600
3000,0.104900
3500,0.079700
4000,0.091800


In [ ]:
model_path = "/content/bert_emotion"
tokenizer_path = "/content/bert_tokenizer"

model.save_pretrained(model_path)
tokenizer.save_pretrained(tokenizer_path)

('/content/bert_tokenizer/tokenizer_config.json',
 '/content/bert_tokenizer/special_tokens_map.json',
 '/content/bert_tokenizer/vocab.txt',
 '/content/bert_tokenizer/added_tokens.json',
 '/content/bert_tokenizer/tokenizer.json')

In [ ]:
import shutil
shutil.make_archive("/content/bert_emotion", 'zip', "/content/bert_emotion")
shutil.make_archive("/content/bert_tokenizer", 'zip', "/content/bert_tokenizer")

'/content/bert_tokenizer.zip'

In [ ]:
trainer.evaluate()

{'eval_loss': 0.2579410970211029,
 'eval_runtime': 3.2509,
 'eval_samples_per_second': 615.207,
 'eval_steps_per_second': 38.45,
 'epoch': 5.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, local_files_only=True)

In [ ]:
labels = [
    "sadness",
    "joy",
    "love",
    "anger",
    "fear",
    "surprise"
]

In [ ]:
def predict_emotion(text):

  inputs = tokenizer(text,return_tensors='pt',truncation=True,padding=True)

  # Move both inputs and model to the same device (cuda)
  inputs = inputs.to("cuda")
  model.to("cuda")

  with torch.no_grad():
    model(**inputs)

  logits = model(**inputs).logits

  predicted_class_id = logits.argmax().item()

  return labels[predicted_class_id]

In [ ]:
text = "I am very sad"

print(predict_emotion(text))

sadness
